<div class="alert alert-block alert-secondary" style="font-size:30px">
EDA Identifying-Age Related Conditions
</div>

<div class="alert alert-block alert-warning" style="font-size:15px">
Hi dear lector 👀, if this notebook is useful for you, please give us an upvote ❤️🚀
    ⚠️🛠️ This notebook is still on work 🛠️⚠️
</div>

<div class="alert alert-block alert-info" style="font-size:15px">
Competition Goals
</div>

According to the competition's description, the main objective is to explore novel approaches in order to enhance the existing methods used in the field. The current methods being employed are the [XGBoost](https://www.kaggle.com/code/alexisbcook/xgboost) and [Random Forest Classifier](https://www.kaggle.com/code/prashant111/random-forest-classifier-tutorial). Therefore, in this scenario, it is essential to comprehend the workings of these methods to gain insight into why they might be experiencing shortcomings.

Both of them are classifiers. In this particular case, we have two cases, namely case 0 and case 1, which represent a binary classification problem. These cases need to be calculated by applying a specific calculation: [probability](https://www.kaggle.com/competitions/icr-identify-age-related-conditions/overview/evaluation):


$$
\text{Log Loss} = -\frac{1}{N_0} \sum_{i=1}^{N_0} y_{0i}\log p_{0i} - \frac{1}{N_1} \sum_{i=1}^{N_1} y_{1i}\log p_{1i}
$$


Helping to discover new mehtods which could be useful for this 
Based on minimal training, you’ll create a model to predict if a person has any of three medical conditions, with an aim to improve on existing methods([Kaggle. (s.f.). ICR](https://www.kaggle.com/competitions/icr-identify-age-related-conditions/overview/description)).


Note: Here you will find code for another notebooks, please, take a look for the references.

# Defining libraries, paths and constants

In [ ]:
!pip install tabpfn --no-index --find-links=file:///kaggle/input/pip-packages-icr/pip-packages
!mkdir -p /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff
!cp /kaggle/input/pip-packages-icr/pip-packages/prior_diff_real_checkpoint_n_0_epoch_100.cpkt /opt/conda/lib/python3.10/site-packages/tabpfn/models_diff/

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import LabelEncoder

# Ploting
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
sns.set_palette('colorblind') ## Better for people with vision problems.
colors = sns.color_palette()
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors)
from tabulate import tabulate

#sklearn
from sklearn.metrics import log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from sklearn.impute import SimpleImputer

#imblearn
import imblearn
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

# Clasifiers
import xgboost
import inspect
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import tensorflow as tf
import tensorflow_decision_forests as tfdf

# Scipy
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from scipy.stats import norm, skew, kurtosis

from collections import defaultdict
from tabpfn import TabPFNClassifier
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

In [ ]:
trainDf = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
greeksDf = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')
testDf = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
mergedDf = pd.merge(trainDf, greeksDf, how = 'left', on = 'Id')

trainDf.columns = trainDf.columns.str.strip()
testDf.columns = testDf.columns.str.strip()
catCols = 'EJ' ## EJ is the unic no numer column, this column is a casificatori one."EJ" is the only non-numeric column in the dataset, and it serves as a categorical variable.

In [ ]:
numColsMerged = mergedDf.columns.tolist()[1:-1]
numColsMerged.remove(catCols)

# Important Notes About the data

<div class="alert alert-block alert-info" style="font-size:15px">
Meaning of features
</div>


In [ ]:
greeksDf.tail(3)

In [ ]:
Alpha = greeksDf['Alpha'].unique()
Beta = greeksDf['Beta'].unique()
Gamma = greeksDf['Gamma'].unique()
Delta = greeksDf['Delta'].unique()

In [ ]:
import matplotlib.pyplot as plt

# Obtener los conteos de valores en las columnas
Alpha = greeksDf['Alpha'].value_counts()
Beta = greeksDf['Beta'].value_counts()
Gamma = greeksDf['Gamma'].value_counts()
Delta = greeksDf['Delta'].value_counts()

fig, axs = plt.subplots(1, 4, figsize=(12, 4))

axs[0].bar(Alpha.index, Alpha.values)
axs[0].set_title('Alpha')

axs[1].bar(Beta.index, Beta.values)
axs[1].set_title('Beta')

axs[2].bar(Gamma.index, Gamma.values)
axs[2].set_title('Gamma')

axs[3].bar(Delta.index, Delta.values)
axs[3].set_title('Delta')

plt.tight_layout()

plt.show()


The data "Greeks" contains same rows, which means we should work with the data merged to help us to se better the big picture.

In [ ]:
len(greeksDf["Id"])

In [ ]:
testDf.head(3)

As it is possible to see, we dont have test data, looking for answers to this, I found the next:

Kaggle officially hides the real test set because the total test set is very small, so even the five samples provided by the official are fake data. The real data will only be provided after you submit the code, and samples are only a method to help you understand whether your submission is correct.([Chen, D](https://www.kaggle.com/competitions/icr-identify-age-related-conditions/discussion/412761))


In [ ]:
len(trainDf["Id"])

Now we know the size of hour dataframe, we have also the same quantitie of rows in greeks Data Frame lets see how many data we have for each case.

In [ ]:
counts = mergedDf['Class'].value_counts()
labels = counts.index.tolist()
sizes = counts.values.tolist()
colors = sns.color_palette('Set3')

# Gráfico de dona
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

ax1.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors, wedgeprops={'width': 0.35})
ax1.axis('equal')

# Gráfico de barras horizontales
y_pos = np.arange(len(labels))
ax2.barh(y_pos, sizes, color=colors)
ax2.set_yticks(y_pos)
ax2.set_yticklabels(labels)
ax2.set_xlabel('Count')

plt.show()


<div class="alert alert-block alert-info" style="font-size:15px">
The devil is in greeks
</div>

Greeks looks like a very strange part of the data, which at first look, seems like data with not sense, afortunately, [@sugataglosh](https://www.kaggle.com/sugataghosh) have helped us to understand this better([Notebook](https://www.kaggle.com/code/sugataghosh/icr-the-devil-is-in-the-greeks)).

Next code only contains little modifications from the original, you cna also check direct the notebook.


In [ ]:
# Min-max normalization
def minmax_scalar(df_train_in, df_valid_in, cols):
    """
    Applies min-max scaling to selected columns
    Args:
        df_train_in (DataFrame, shape (m, n)): input training dataframe
        df_valid_in (DataFrame, shape (m, n)): input validation dataframe
        cols (array_like, shape (r, ))       : list of columns to be normalized (r <= n)
        
    Returns:
        df_train_out (DataFrame, shape (m, n)): output training dataframe
        df_valid_out (DataFrame, shape (m, n)): output validation dataframe
    """
    df_train_out, df_valid_out = df_train_in.copy(deep = True), df_valid_in.copy(deep = True)
    cols = [col for col in cols if col in df_train_in.columns]
    cols = [col for col in cols if df_train_in[col].nunique() > 1]
    for col in cols:
        min_, max_ = df_train_out[col].min(), df_train_out[col].max()
        df_train_out[col] = (df_train_out[col] - min_) / (max_ - min_)
        df_valid_out[col] = (df_valid_out[col] - min_) / (max_ - min_)
    return df_train_out, df_valid_out

In [ ]:
# Cross validation scores
def cv_scores(model, X, y, n_splits = 5, scaling = []):
    """
    Function to return cross validation log loss scores along with mean and standard deviation
    Args:
        model                            : untrained model
        X (DataFrame, shape (m, n))      : feature dataframe
        y (Series, shape (m, ))          : target variable
        n_splits (scalar)                : number of folds for cross validation
        scaling (array_like, shape (r, )): list of columns of X to be scaled (r <= n)
        
    Returns:
        scores (array_like, shape (n_splits, )): list of cross validation log loss scores
        mean (scalar): mean of the cross validation log loss scores
        std (scalar) : standard deviation of the cross validation log loss scores
    """
    scores = []
    skf = StratifiedKFold(n_splits = n_splits, shuffle = True, random_state = 0)
    for train_idx, val_idx in skf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_valid, y_valid = X.iloc[val_idx], y.iloc[val_idx]
        X_train, X_valid = minmax_scalar(X_train, X_valid, scaling)
        if 'CatBoostClassifier' in str(model):
            model.fit(X_train, y_train, eval_set = [(X_valid, y_valid)], verbose = 0)
        else:
            model.fit(X_train, y_train)
        val_preds = model.predict_proba(X_valid)
        val_score = log_loss(y_valid, val_preds)
        scores.append(val_score)
    mean, std = np.mean(scores), np.std(scores)
    return scores, mean, std

In [ ]:
# Categorical data encoding
le = LabelEncoder()
categorical_columns = mergedDf.columns[mergedDf.dtypes == 'object']
categorical_columns = [col for col in categorical_columns if col not in ['Id', 'Epsilon']]

mergedDf[categorical_columns] = mergedDf[categorical_columns].apply(lambda x: le.fit_transform(x))
mergedDf[categorical_columns].head(3)


In [ ]:
# Columns to be scaled
scl = [col for col in mergedDf.columns if col not in ['Id', 'Class', 'Epsilon']]
scl = [col for col in scl if mergedDf[col].dtypes == 'float64']

We will use these features for comparison in another section of the notebook, as we are starting to gain a better understanding of our data.

<div class="alert alert-block alert-info" style="font-size:15px">
Finding important Features
</div>

The features in this case are totally unknown of meaning, but it is still important to have an idea of the range each of them works, for that reason we need to find the ones which have things in common. We also know that our data is taked from humans,so begin for that way.

In [ ]:
featuresStd = mergedDf.loc[:,numColsMerged].apply(lambda x: np.std(x)).sort_values(
    ascending=False)
f_std = mergedDf[featuresStd.iloc[:20].index.tolist()]
f_std

Next part of code is based on this [Notebook](https://www.kaggle.com/code/datafan07/icr-simple-eda-baseline), I just make some ligth changes to finding out the behaiviour with merged Dataframe

In [ ]:
featuresStd = mergedDf.loc[:,numColsMerged].apply(lambda x: np.std(x)).sort_values(
    ascending=False)
fStd = mergedDf[featuresStd.iloc[:20].index.tolist()]

with pd.option_context('mode.use_inf_as_na', True):
    featuresSkew = np.abs(mergedDf.loc[:,numColsMerged].apply(lambda x: np.abs(skew(x))).sort_values(
        ascending=False)).dropna()
skewed = mergedDf[featuresSkew.iloc[:20].index.tolist()]

with pd.option_context('mode.use_inf_as_na', True):
    featuresKurt = np.abs(mergedDf.loc[:,numColsMerged].apply(lambda x: np.abs(kurtosis(x))).sort_values(
        ascending=False)).dropna()
kurtF = mergedDf[featuresKurt.iloc[:20].index.tolist()]

In [ ]:
def feat_dist(df, cols, rows=3, columns=3, title=None, figsize=(30, 25)):
    
    fig, axes = plt.subplots(rows, columns, figsize=figsize, constrained_layout=True)
    axes = axes.flatten()

    for i, j in zip(cols, axes):
        sns.kdeplot(df, x=i, ax=j, hue='Class', linewidth=1.5, linestyle='--')
        
        (mu, sigma) = norm.fit(df[i])
        
        xmin, xmax = j.get_xlim()[0], j.get_xlim()[1]
        x = np.linspace(xmin, xmax, 100)
        p = norm.pdf(x, mu, sigma)
        j.plot(x, p, 'k', linewidth=2)
        
        if 'Skewed' in title:
            label = 'Skewed'
        elif 'Kurtosis' in title:
            label = 'Kurtosis'
        elif 'Desviation Standard' in title:
            label = 'Desviation Standard'
        else:
            label = 'Normal Dist'
        
        j.set_title('Dist of {0} Norm Fit: $\mu=${1:.2g}, $\sigma=${2:.2f}'.format(i, mu, sigma), weight='bold')
        j.legend(labels=[f'Class0_{i}', f'Class1_{i}', label])
        fig.suptitle(f'{title}', fontsize=24, weight='bold')

In this section, we will explore some selected features and examine the characteristics displayed by the standard deviation , [Kurtosis](https://www.scribbr.com/statistics/kurtosis/) and [Skewed](https://www.scribbr.com/statistics/skewness/).


**Kurtosis Lecture:**

<img src="https://www.scribbr.com/wp-content/uploads/2022/07/The-difference-between-skewness-and-kurtosis.webp" alt="The difference between skewness and kurtosis" width="400">

**Skewed Lecture:**
![Skewness of a distribution](https://www.scribbr.com/wp-content/uploads/2022/05/Skewness-of-a-distribution.webp)


In [ ]:
feat_dist(mergedDf, f_std.columns.tolist(), rows=2, columns=4, title='High Desviation Standard Features', figsize=(30, 9))

In [ ]:
feat_dist(mergedDf, skewed.columns.tolist(), rows=2, columns=4, title='Distribution of Skewed Features', figsize=(30, 9))

In [ ]:
feat_dist(mergedDf, kurtF.columns.tolist(), rows=2, columns=4, title='Distribution of High Kurtosis Features', figsize=(30, 9))

These charts help identify the best aligned features between classes, which is crucial. It aids in detecting potential overfitting issues by identifying removable features. Alternatively, if these aligned features offer valuable insights for binary classification, they can be retained based on which method did you choose.

The kurtosis analysis is particularly interesting, revealing pronounced tailedness in class 1. This suggests the presence of outliers and extreme values, impacting classification outcomes. Awareness of such distributions enables us to adapt modeling techniques, making informed decisions.

In [ ]:
correlations = mergedDf.loc[:,numColsMerged].corrwith(mergedDf['Class']).to_frame()
correlations['Abs Corr'] = correlations[0].abs()
sorted_correlations = correlations.sort_values('Abs Corr', ascending=False)['Abs Corr']
fig, ax = plt.subplots(figsize=(6,4))
sns.heatmap(sorted_correlations.iloc[1:].to_frame()[sorted_correlations>=.15], cmap='inferno', annot=True, vmin=-1, vmax=1, ax=ax)
plt.title('Feature Correlations With Target')
plt.show()

Alpha and Gamma show a high correlation with the Class variable in the dataset. Thus, we need to explore different methods and combinations of the data, along with other possible approaches, but due to this high correlation we could know that this is a very important part of the dataset.

In [ ]:
correlations = mergedDf.loc[:,numColsMerged].corr().abs().unstack().sort_values(kind="quicksort",ascending=False).reset_index()
correlations = correlations[correlations['level_0'] != correlations['level_1']] #preventing 1.0 corr
corr_max=correlations.level_0.head(150).tolist()
corr_max=list(set(corr_max)) #removing duplicates

corr_min=correlations.level_0.tail(34).tolist()
corr_min=list(set(corr_min)) #removing duplicates
correlation_train = mergedDf.loc[:,corr_max].corr()

In [ ]:
mask = np.triu(correlation_train.corr())

plt.figure(figsize=(30, 12))
sns.heatmap(correlation_train,
            mask=mask,
            annot=True,
            fmt='.3f',
            cmap='coolwarm',
            linewidths=0.00,
            cbar=True)


plt.suptitle('Features with Highest Correlations',  weight='bold')
plt.tight_layout()

By analyzing the heatmap, we can observe some intriguing correlations. As previously noted, Alpha and Gamma exhibit a strong positive correlation with the "Class" variable. However, it is interesting to note that Alpha and Gamma also demonstrate negative correlations with other variables. Furthermore, the heatmap reaffirms the presence of significant correlations previously identified through the distribution plots.

In [ ]:
import os
filepath = '/kaggle/input/icr-identify-age-related-conditions'

In [ ]:
correlations = mergedDf.loc[:,numColsMerged].corr().abs().unstack().reset_index()
correlations

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12,4), constrained_layout=True)
axes = axes.flatten()

# for i, j in zip(cols, axes):
i = 0
for row in range(0,16,2):
    a = correlations.reset_index(drop=True).loc[row, ['level_0', 'level_1']][0]
    b = correlations.reset_index(drop=True).loc[row, ['level_0', 'level_1']][1]    
   
    sns.regplot(mergedDf, x=a, y=b, ci=False, ax=axes[i], order=1, scatter_kws={'color':'red', 's':1.5}, line_kws={'color':'black', 'linewidth':1.5})
    i+=1
    
plt.suptitle('Highly Correlated Features',  weight='bold')
plt.show()

# "Types of data"

There are some columns that might represent age, but we are not certain. Additionally, we could explore the columns that potentially involve percentages. To gain insights, let's examine the relationships using a dendrogram and filter out certain columns from the dataset.

In [ ]:
trainDf['EJ'] = trainDf['EJ'].replace({'A': 0, 'B': 1})

In [ ]:
def hierarchical_clustering(data):
    fig, ax = plt.subplots(1, 1, figsize=(18, 10), dpi=120)
    correlations = data.corr()
    converted_corr = 1 - np.abs(correlations)
    Z = linkage(squareform(converted_corr), 'complete')
    
    dn = dendrogram(Z, labels=data.columns, ax=ax, above_threshold_color='#ff0000', orientation='right')
    hierarchy.set_link_color_palette(['#000000'])  # Change color of lines to black
    plt.grid(axis='x')
    plt.title('Hierarchical Clustering Dendrogram', fontsize=18, fontweight='bold')
    plt.xlabel('Columns')
    plt.ylabel('Distance')
    plt.xticks(rotation=90)  # Rotate the column names for better readability.
    plt.show()

hierarchical_clustering(trainDf.drop(['Class', 'Id'], axis=1))


In general, the dendrogram provides an effective visual representation of the relationships between columns in a dataset, which can be useful for exploring the structure and discovering patterns in the data. In our case, it would help us better understand what we are working with. However, it is necessary to contrast the available information and seek the assistance of medical professionals to try to identify or at least make assumptions about the underlying conditions or factors.

<div class="alert alert-block alert-info" style="font-size:15px">
Lets check data between 0 and 1</div>

In [ ]:
numericColumns = trainDf.select_dtypes(include=[np.number])
numericColumns = numericColumns.apply(pd.to_numeric, errors='coerce')
filteredData = numericColumns.loc[:, (numericColumns >= 0).all()]
filteredData = numericColumns[(numericColumns >= 0) & (numericColumns <= 1)]
filteredData = filteredData.dropna(axis=1, how='all')

In [ ]:
pd.set_option('display.max_columns', None)
filteredData.head(2)

<div class="alert alert-block alert-info" style="font-size:15px">
Lets check data between 59 and 100</div>

In [ ]:
numericColumns = trainDf.select_dtypes(include=[np.number])
numericColumns = numericColumns.apply(pd.to_numeric, errors='coerce')

filteredData = numericColumns.loc[:, (numericColumns >= 50).all()]
filteredData = filteredData.loc[:, (filteredData <= 100).all()]
filteredData = numericColumns[(numericColumns >= 50) & (numericColumns <= 100)] ## Aqui estoy intentando ver si existe alguna feature relacionada con la edad
filteredData =  filteredData.dropna(axis=1, how='all')


In [ ]:
filteredData.head(2)

Dendogram give us better information about what features have a better relation, in any way, just making filters for the range of data, doesn't seems to util as the resting columns are too many, even columns with have been resalted as the posible "Age" feature like GH have multiple empties values after filtering.

# Training aproximations

In this part of the notebook I'm just observing different models which have been already used, I'm not trying to get a great score, but also it is important to say, as we have data in which we have no idea what is about, we can also try modifyng the data before each training in differents ways, like deleting some columns, or taking away some decimals, so I reccommend to explore the data frame a do a lots of tests.

# Devil is in greeks Trainings

<div class="alert alert-block alert-info" style="font-size:15px">
The devil is in greeks Trainings
</div>

<div class="alert alert-block alert-success" style="font-size:15px">

Prediction based on train dataset

</div>


In [ ]:
# Features target split
features = trainDf.drop(['Id', 'Class', 'BD', 'CD', 'CW', 'FD'], axis = 1).columns.tolist()
X, y = mergedDf[features], mergedDf['Class']

In [ ]:
# XGBoost

xgb = XGBClassifier(n_jobs=-1)
xgb_scores_t, xgb_mean_t, xgb_std_t = cv_scores(xgb, X, y, n_splits=5, scaling=scl)
data = [['XGBoost', xgb_scores_t, xgb_mean_t, xgb_std_t]]
headers = ['Model', 'Scores', 'Mean', 'Std.Dev.']
dfXgb_scores_t = pd.DataFrame(data, columns=headers)
dfXgb_scores_t.head()

In [ ]:
# CatBoost
catb = CatBoostClassifier()
catb_scores_t, catb_mean_t, catb_std_t = cv_scores(catb, X, y, n_splits=5, scaling=scl)
data = [['CatBoost', catb_scores_t, catb_mean_t, catb_std_t]]
headers = ['Model', 'Scores', 'Mean', 'Std.Dev.']
dfCatb = pd.DataFrame(data, columns=headers)
dfCatb.head()

<div class="alert alert-block alert-success" style="font-size:15px">

Prediction based on merged dataset

</div>

In [ ]:
# Features target split
features = mergedDf.drop(['Id', 'Class', 'Alpha', 'Epsilon'], axis = 1).columns.tolist()
X, y = mergedDf[features], mergedDf['Class']

In [ ]:
# XGBoost
xgb = XGBClassifier(n_jobs=-1)
xgb_scores_tg, xgb_mean_tg, xgb_std_tg = cv_scores(xgb, X, y, n_splits=5, scaling=scl)
model = "XGBoost"
headers = ["Model", "Scores", "Mean", "Std.Dev."]
data = [[model, xgb_scores_tg, xgb_mean_tg, xgb_std_tg]]
dfXgb_scores_tg = pd.DataFrame(data, columns=headers)
dfXgb_scores_tg.head()

In [ ]:
# CatBoost
catb = CatBoostClassifier()
catb_scores_tg, catb_mean_tg, catb_std_tg = cv_scores(catb, X, y, n_splits=5, scaling=scl)
model = "CatBoost"
headers = ["Model", "Scores", "Mean", "Std.Dev."]
data = [[model, catb_scores_tg, catb_mean_tg, catb_std_tg]]
dfCatb_scores_tg = pd.DataFrame(data, columns=headers)
dfCatb_scores_tg.head()

These predictions are somewhat interesting, but we can strive for better approximations by exploring alternative models. Although the original notebook included LightGBM, I opted to exclude it.

# Tensorflow tenplates

<div class="alert alert-block alert-info" style="font-size:15px">
Tensorflow templates
</div>

This part of the code comes from this [notebook](https://www.kaggle.com/code/gusthema/identifying-age-related-conditions-w-tfdf#Quick-basic-dataset-exploration)

In [ ]:
## Lets reset trainDf
trainDf = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')

Getting the Columns which we want work with.

In [ ]:
FEATURE_COLUMNS = [i for i in trainDf.columns if i not in ["Id"]]

In this part we are spliting the data with kfolg, KFold is a K-Fold cross-validator that provides training/test indices to split data into training/test sets. It divides the dataset into k consecutive folds (without shuffling by default). Each fold is used once as a validation while the remaining k-1 folds form the training set.

In [ ]:
# Creates a GroupKFold with 5 splits
kf = KFold(n_splits=5)

During the k-fold cross-validation process, predictions are generated on test sets that consist of data not utilized for training the model. These predictions, known as out-of-fold predictions, play a significant role in machine learning. They serve as an essential component for estimating a model's performance when making predictions on unseen data in the future. Furthermore, out-of-fold predictions contribute to the development of ensemble models, enhancing their overall effectiveness. [read more](https://machinelearningmastery.com/out-of-fold-predictions-in-machine-learning/)

In [ ]:
# Create list of ids for the creation of oof dataframe.
ID_LIST = trainDf.index

# Create a dataframe of required size with zero values.
oof = pd.DataFrame(data=np.zeros((len(ID_LIST),1)), index=ID_LIST)

# Create an empty dictionary to store the models trained for each fold.
models = {}

# Create empty dict to save metircs for the models trained for each fold.
accuracy = {}
cross_entropy = {}

# Save the name of the label column to a variable.
label = "Class"

Next we are getting models, for this case, Random Forest models were chosen to be used in Keras. However, it should be noted that the selection of Random Forest does not necessarily imply that it is the best-performing model.

In [ ]:
tfdf.keras.get_all_models()

In [ ]:
# Calculate the number of negative and positive values in `Class` column
neg, pos = np.bincount(trainDf['Class'])
# Calculate total samples
total = neg + pos
print('Examples:\n    Total: {}\n    Positive: {} ({:.2f}% of total)\n'.format(
    total, pos, 100 * pos / total))

## Hyperparameter tuning to avoid overfitting

Because of the smaller size of the dataset, it is likely that the model will overfit during training. Numerous parameters, primarily max_depth and num_trees can be changed to fine-tune the model and prevent overfitting.

The attributemax_depth indicates the maximum depth of the tree. To avoid overfitting, we can try to reduce the depth of the tree from it's default value, which is 16. Another way to tackle overfitting is to increase the number of individual decision trees. To do this, we have to increase the value of the parameter num_trees from its default value(300).

You can set these parameters as follows:

rf = tfdf.keras.RandomForestModel(max_depth=5, num_trees=500)

It is also possible to construct a function that tests different depths and selects the best one after multiple training iterations.

Between the two options, Undersampling and Class Weighting, the latter was chosen for this case. It is important to note that while it may yield good results, it could introduce biases when compared to the unseen data in the competition.

In [ ]:
# Calculate the number of samples for each label.
neg, pos = np.bincount(trainDf['Class'])

# Calculate total samples.
total = neg + pos

# Calculate the weight for each label.
weight_for_0 = (1 / neg) * (total / 2.0)
weight_for_1 = (1 / pos) * (total / 2.0)

class_weight = {0: weight_for_0, 1: weight_for_1}

print('Weight for class 0: {:.2f}'.format(weight_for_0))
print('Weight for class 1: {:.2f}'.format(weight_for_1))



To train and evaluate the models using class weights, use the dict in model.fit() as an argument as shown below.

model.fit(x=train_ds, class_weight=class_weight)


## Train Random Forest Model

Today, we will use the defaults to create the Random Forest Model. By default the model is set to train for a classification task. We will train a model for each fold and after training we will store the model and metrics. Here, we have chosen accuracy and binary_crossentropy as the metrics.

In [ ]:
# Loop through each fold
for i, (train_index, valid_index) in enumerate(kf.split(X=trainDf)):
        print('##### Fold',i+1)

        # Fetch values corresponding to the index 
        train_df = trainDf.iloc[train_index]
        valid_df = trainDf.iloc[valid_index]
        valid_ids = valid_df.index.values
        
        # Select only feature columns for training.
        train_df = train_df[FEATURE_COLUMNS]
        valid_df = valid_df[FEATURE_COLUMNS]
        
        # There's one more step required before we can train the model. 
        # We need to convert the datatset from Pandas format (pd.DataFrame)
        # into TensorFlow Datasets format (tf.data.Dataset).
        # TensorFlow Datasets is a high performance data loading library 
        # which is helpful when training neural networks with accelerators like GPUs and TPUs.
        # Note: Some column names contains white spaces at the end of their name, 
        # which is non-comaptible with SavedModels save format. 
        # By default, `pd_dataframe_to_tf_dataset` function will convert 
        # this column names into a compatible format. 
        # So you can safely ignore the warnings related to this.
        train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df, label=label)
        valid_ds = tfdf.keras.pd_dataframe_to_tf_dataset(valid_df, label=label)

        # Define the model and metrics
        rf = tfdf.keras.RandomForestModel()
        rf.compile(metrics=["accuracy", "binary_crossentropy"]) 
        
        # Train the model
        # We will train the model using a one-liner.
        # Note: you may see a warning about Autograph. 
        # You can safely ignore this, it will be fixed in the next release.
        # Previously calculated class weights is used to handle imbalance.
        rf.fit(x=train_ds, class_weight=class_weight)
        
        # Store the model
        models[f"fold_{i+1}"] = rf
        
        
        # Predict OOF value for validation data
        predict = rf.predict(x=valid_ds)
        
        # Store the predictions in oof dataframe
        oof.loc[valid_ids, 0] = predict.flatten() 
        
        # Evaluate and store the metrics in respective dicts
        evaluation = rf.evaluate(x=valid_ds,return_dict=True)
        accuracy[f"fold_{i+1}"] = evaluation["accuracy"]
        cross_entropy[f"fold_{i+1}"]= evaluation["binary_crossentropy"]

## Visualize model

In [ ]:
tfdf.model_plotter.plot_model_in_colab(models['fold_1'], tree_idx=0, max_depth=3)

## Evaluate Model

In [ ]:
figure, axis = plt.subplots(3, 2, figsize=(10, 10))
plt.subplots_adjust(hspace=0.5, wspace=0.3)

for i, fold_no in enumerate(models.keys()):
    row = i//2
    col = i % 2
    logs = models[fold_no].make_inspector().training_logs()
    axis[row, col].plot([log.num_trees for log in logs], [log.evaluation.loss for log in logs])
    axis[row, col].set_title(f"Fold {i+1}")
    axis[row, col].set_xlabel('Number of trees')
    axis[row, col].set_ylabel('Loss (out-of-bag)')

axis[2][1].set_visible(False)
plt.show()

In [ ]:
for _model in models:
    inspector = models[_model].make_inspector()
    print(_model, inspector.evaluation())

In [ ]:
average_loss = 0
average_acc = 0

for _model in  models:
    average_loss += cross_entropy[_model]
    average_acc += accuracy[_model]
    print(f"{_model}: acc: {accuracy[_model]:.4f} loss: {cross_entropy[_model]:.4f}")

print(f"\nAverage accuracy: {average_acc/5:.4f}  Average loss: {average_loss/5:.4f}")

In [ ]:
inspector = models['fold_1'].make_inspector()

print(f"Available variable importances:")
for importance in inspector.variable_importances().keys():
  print("\t", importance)

In [ ]:
# Each line is: (feature name, (index of the feature), importance score)
inspector.variable_importances()["NUM_AS_ROOT"]

For this model, I also tried using the dataframe MergedDf, but the results came with overfitting. In this notebook, my intention is not to achieve better results, but rather to compare the models already used in the competition. I'm adding some comparisons and possible combinations for dataframes in the next versions.

# Public_krni_pdi_

This part comes from this [notebook](https://www.kaggle.com/code/aikhmelnytskyy/public-krni-pdi-with-two-additional-models)

In [ ]:
train = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/train.csv')
test = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/test.csv')
sample = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/sample_submission.csv')
greeks = pd.read_csv('/kaggle/input/icr-identify-age-related-conditions/greeks.csv')

In [ ]:
# lb = LabelEncoder()
# train['EJ'] = lb.fit_transform(train['EJ']).astype(float)
# test['EJ'] = lb.fit_transform(test['EJ']).astype(float)

In [ ]:
first_category = train.EJ.unique()[0]
train.EJ = train.EJ.eq(first_category).astype('int')
test.EJ = test.EJ.eq(first_category).astype('int')

In [ ]:
def random_under_sampler(df):
    # Calculate the number of samples for each label. 
    neg, pos = np.bincount(df['Class'])

    # Choose the samples with class label `1`.
    one_df = df.loc[df['Class'] == 1] 
    # Choose the samples with class label `0`.
    zero_df = df.loc[df['Class'] == 0]
    # Select `pos` number of negative samples.
    # This makes sure that we have equal number of samples for each label.
    zero_df = zero_df.sample(n=pos)

    # Join both label dataframes.
    undersampled_df = pd.concat([zero_df, one_df])

    # Shuffle the data and return
    return undersampled_df.sample(frac = 1)

In [ ]:
train_good = random_under_sampler(train)

In [ ]:
train_good.shape

In [ ]:
predictor_columns = [n for n in train.columns if n != 'Class' and n != 'Id']
x= train[predictor_columns]
y = train['Class']

In [ ]:
# x_norm = np.array(x_norm)
# y_ros = np.array(y_ros)

In [ ]:
from sklearn.model_selection import KFold as KF, GridSearchCV
cv_outer = KF(n_splits = 10, shuffle=True, random_state=42)
cv_inner = KF(n_splits = 5, shuffle=True, random_state=42)

In [ ]:
def balanced_log_loss(y_true, y_pred):
    # y_true: correct labels 0, 1
    # y_pred: predicted probabilities of class=1
    # calculate the number of observations for each class
    N_0 = np.sum(1 - y_true)
    N_1 = np.sum(y_true)
    # calculate the weights for each class to balance classes
    w_0 = 1 / N_0
    w_1 = 1 / N_1
    # calculate the predicted probabilities for each class
    p_1 = np.clip(y_pred, 1e-15, 1 - 1e-15)
    p_0 = 1 - p_1
    # calculate the summed log loss for each class
    log_loss_0 = -np.sum((1 - y_true) * np.log(p_0))
    log_loss_1 = -np.sum(y_true * np.log(p_1))
    # calculate the weighted summed logarithmic loss
    # (factgor of 2 included to give same result as LL with balanced input)
    balanced_log_loss = 2*(w_0 * log_loss_0 + w_1 * log_loss_1) / (w_0 + w_1)
    # return the average log loss
    return balanced_log_loss/(N_0+N_1)

In [ ]:
class Ensemble():
    def __init__(self):
        self.imputer = SimpleImputer(missing_values=np.nan, strategy='median')

        self.classifiers =[xgboost.XGBClassifier(n_estimators=100,max_depth=3,learning_rate=0.2,subsample=0.9,colsample_bytree=0.85),
                           xgboost.XGBClassifier(),
                           TabPFNClassifier(N_ensemble_configurations=24),
                          TabPFNClassifier(N_ensemble_configurations=64)]
    
    def fit(self,X,y):
        y = y.values
        unique_classes, y = np.unique(y, return_inverse=True)
        self.classes_ = unique_classes
        first_category = X.EJ.unique()[0]
        X.EJ = X.EJ.eq(first_category).astype('int')
        X = self.imputer.fit_transform(X)
#         X = normalize(X,axis=0)
        for classifier in self.classifiers:
            if classifier==self.classifiers[2] or classifier==self.classifiers[3]:
                classifier.fit(X,y,overwrite_warning =True)
            else :
                classifier.fit(X, y)
     
    def predict_proba(self, x):
        x = self.imputer.transform(x)
#         x = normalize(x,axis=0)
        probabilities = np.stack([classifier.predict_proba(x) for classifier in self.classifiers])
        averaged_probabilities = np.mean(probabilities, axis=0)
        class_0_est_instances = averaged_probabilities[:, 0].sum()
        others_est_instances = averaged_probabilities[:, 1:].sum()
        # Weighted probabilities based on class imbalance
        new_probabilities = averaged_probabilities * np.array([[1/(class_0_est_instances if i==0 else others_est_instances) for i in range(averaged_probabilities.shape[1])]])
        return new_probabilities / np.sum(new_probabilities, axis=1, keepdims=1) 

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
def training(model, x,y,y_meta):
    outer_results = list()
    best_loss = np.inf
    split = 0
    splits = 5
    for train_idx,val_idx in tqdm(cv_inner.split(x), total = splits):
        split+=1
        x_train, x_val = x.iloc[train_idx],x.iloc[val_idx]
        y_train, y_val = y_meta.iloc[train_idx], y.iloc[val_idx]
                
        model.fit(x_train, y_train)
        y_pred = model.predict_proba(x_val)
        probabilities = np.concatenate((y_pred[:,:1], np.sum(y_pred[:,1:], 1, keepdims=True)), axis=1)
        p0 = probabilities[:,:1]
        p0[p0 > 0.86] = 1
        p0[p0 < 0.14] = 0
        y_p = np.empty((y_pred.shape[0],))
        for i in range(y_pred.shape[0]):
            if p0[i]>=0.5:
                y_p[i]= False
            else :
                y_p[i]=True
        y_p = y_p.astype(int)
        loss = balanced_log_loss(y_val,y_p)

        if loss<best_loss:
            best_model = model
            best_loss = loss
            print('best_model_saved')
        outer_results.append(loss)
        print('>val_loss=%.5f, split = %.1f' % (loss,split))
    print('LOSS: %.5f' % (np.mean(outer_results)))
    return best_model
    

In [ ]:
from datetime import datetime
times = greeks.Epsilon.copy()
times[greeks.Epsilon != 'Unknown'] = greeks.Epsilon[greeks.Epsilon != 'Unknown'].map(lambda x: datetime.strptime(x,'%m/%d/%Y').toordinal())
times[greeks.Epsilon == 'Unknown'] = np.nan

In [ ]:
train_pred_and_time = pd.concat((train, times), axis=1)
test_predictors = test[predictor_columns]
first_category = test_predictors.EJ.unique()[0]
test_predictors.EJ = test_predictors.EJ.eq(first_category).astype('int')
test_pred_and_time = np.concatenate((test_predictors, np.zeros((len(test_predictors), 1)) + train_pred_and_time.Epsilon.max() + 1), axis=1)

In [ ]:
ros = RandomOverSampler(random_state=42)

train_ros, y_ros = ros.fit_resample(train_pred_and_time, greeks.Alpha)
print('Original dataset shape')
print(greeks.Alpha.value_counts())
print('Resample dataset shape')
print( y_ros.value_counts())

In [ ]:
x_ros = train_ros.drop(['Class', 'Id'],axis=1)
y_ = train_ros.Class

In [ ]:
yt = Ensemble()

In [ ]:
m = training(yt,x_ros,y_,y_ros)

In [ ]:
y_.value_counts()/y_.shape[0]

In [ ]:
y_pred = m.predict_proba(test_pred_and_time)
probabilities = np.concatenate((y_pred[:,:1], np.sum(y_pred[:,1:], 1, keepdims=True)), axis=1)
p0 = probabilities[:,:1]
p0[p0 > 0.74] = 1
p0[p0 < 0.26] = 0

# Submission

In [ ]:
submission = pd.DataFrame(test["Id"], columns=["Id"])
submission["class_0"] = p0
submission["class_1"] = 1 - p0
submission.to_csv('submission.csv', index=False)

In [ ]:
submission_df = pd.read_csv('submission.csv')
submission_df